# Healthcare Patient Clustering by Clinical Indicators

**Dataset:** Diabetes Health Indicators Dataset (CDC BRFSS 2015) — Kaggle (`alexteboul/diabetes-health-indicators-dataset`)

**Goal:** Group patients into clinically meaningful risk segments using unsupervised learning, without using the diabetes label during training.

**Pipeline**

```
PATIENT DATASET -> Data Exploration -> Missing Value Handling -> Outlier Investigation
-> Clinical Feature Selection -> Encoding + Scaling
-> K-Means  |  Hierarchical Clustering
-> Cluster Evaluation (Silhouette / Davies-Bouldin / Calinski-Harabasz)
-> Best Clustering -> Cluster Profiling -> Clinical Interpretation
-> Visualization -> Healthcare Insights
```

## 1. Imports and Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)
np.random.seed(42)

os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

### Dataset download

The dataset was pulled from Kaggle into `data/`. To re-download it:

```bash
# Option A: Kaggle CLI (needs ~/.kaggle/kaggle.json)
kaggle datasets download -d alexteboul/diabetes-health-indicators-dataset -p data --unzip

# Option B: direct endpoint
curl -L -o data/kaggle_diabetes.zip \
  "https://www.kaggle.com/api/v1/datasets/download/alexteboul/diabetes-health-indicators-dataset"
unzip -o data/kaggle_diabetes.zip -d data
```

We use `diabetes_012_health_indicators_BRFSS2015.csv`, which keeps the 3-level target
(0 = no diabetes, 1 = prediabetes, 2 = diabetes). The target is **held out** from clustering and
only used at the end to validate the clinical meaning of each cluster.

In [2]:
df = pd.read_csv("data/diabetes_012_health_indicators_BRFSS2015.csv")
print("Rows, Columns:", df.shape)
df.head()

Rows, Columns: (253680, 22)


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


## 2. Data Exploration

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_012          253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Diabetes_012,253680.0,0.296921,0.698160,0.0,0.0,0.0,0.0,2.0
HighBP,253680.0,0.429001,0.494934,0.0,0.0,0.0,1.0,1.0
HighChol,253680.0,0.424121,0.494210,0.0,0.0,0.0,1.0,1.0
CholCheck,253680.0,0.962670,0.189571,0.0,1.0,1.0,1.0,1.0
BMI,253680.0,28.382364,6.608694,12.0,24.0,27.0,31.0,98.0
Smoker,253680.0,0.443169,0.496761,0.0,0.0,0.0,1.0,1.0
Stroke,253680.0,0.040571,0.197294,0.0,0.0,0.0,0.0,1.0
HeartDiseaseorAttack,253680.0,0.094186,0.292087,0.0,0.0,0.0,0.0,1.0
PhysActivity,253680.0,0.756544,0.429169,0.0,1.0,1.0,1.0,1.0
Fruits,253680.0,0.634256,0.481639,0.0,0.0,1.0,1.0,1.0


In [5]:
# Target distribution (held out from clustering, used only for validation later)
print(df["Diabetes_012"].value_counts())
print()
print((df["Diabetes_012"].value_counts(normalize=True) * 100).round(2))

plt.figure(figsize=(6, 4))
sns.countplot(x="Diabetes_012", data=df, palette="viridis")
plt.xticks([0, 1, 2], ["No Diabetes", "Prediabetes", "Diabetes"])
plt.title("Diabetes Class Distribution")
plt.ylabel("Patients")
plt.show()

Diabetes_012
0.0    213703
2.0     35346
1.0      4631
Name: count, dtype: int64

Diabetes_012
0.0    84.24
2.0    13.93
1.0     1.83
Name: proportion, dtype: float64


/tmp/ipykernel_29795/699055748.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x="Diabetes_012", data=df, palette="viridis")
/tmp/ipykernel_29795/699055748.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Distribution of the continuous-ish clinical variables
cont_cols = ["BMI", "GenHlth", "MentHlth", "PhysHlth", "Age"]
df[cont_cols].hist(bins=30, figsize=(14, 8), color="teal")
plt.suptitle("Distribution of Continuous Clinical Indicators")
plt.tight_layout()
plt.show()

/tmp/ipykernel_29795/1941015581.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Prevalence of each binary clinical condition
bin_cols = ["HighBP", "HighChol", "CholCheck", "Smoker", "Stroke", "HeartDiseaseorAttack",
            "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump", "AnyHealthcare",
            "NoDocbcCost", "DiffWalk", "Sex"]

prev = (df[bin_cols].mean() * 100).sort_values(ascending=False)
print(prev.round(2))

plt.figure(figsize=(10, 5))
prev.plot(kind="bar", color="steelblue")
plt.title("Prevalence of Binary Indicators (% of patients = 1)")
plt.ylabel("% of patients")
plt.tight_layout()
plt.show()

CholCheck               96.27
AnyHealthcare           95.11
Veggies                 81.14
PhysActivity            75.65
Fruits                  63.43
Smoker                  44.32
Sex                     44.03
HighBP                  42.90
HighChol                42.41
DiffWalk                16.82
HeartDiseaseorAttack     9.42
NoDocbcCost              8.42
HvyAlcoholConsump        5.62
Stroke                   4.06
dtype: float64


/tmp/ipykernel_29795/1666998697.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Correlation heatmap of all variables
plt.figure(figsize=(14, 11))
sns.heatmap(df.corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Matrix - All Variables")
plt.tight_layout()
plt.show()

# Which variables move most with diabetes status?
print(df.corr()["Diabetes_012"].drop("Diabetes_012").sort_values(ascending=False).round(3))

/tmp/ipykernel_29795/1928424351.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


GenHlth                 0.303
HighBP                  0.272
BMI                     0.224
DiffWalk                0.224
HighChol                0.209
Age                     0.185
HeartDiseaseorAttack    0.180
PhysHlth                0.176
Stroke                  0.107
MentHlth                0.074
CholCheck               0.068
Smoker                  0.063
NoDocbcCost             0.035
Sex                     0.031
AnyHealthcare           0.015
Fruits                 -0.042
HvyAlcoholConsump      -0.058
Veggies                -0.059
PhysActivity           -0.122
Education              -0.131
Income                 -0.171
Name: Diabetes_012, dtype: float64


## 3. Missing Value Handling

In [9]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Total missing:", df.isnull().sum().sum())

Missing values per column:
Diabetes_012            0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

Total missing: 0


In [10]:
# No nulls in this dataset, but duplicate survey rows exist - they bias cluster centroids
# toward the most common response pattern, so we drop them.
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)

Duplicate rows: 23899
Shape after dropping duplicates: (229781, 22)


In [11]:
# Sanity check on impossible / out-of-range clinical values
print("BMI range:", df["BMI"].min(), "-", df["BMI"].max())
print("MentHlth range:", df["MentHlth"].min(), "-", df["MentHlth"].max())
print("PhysHlth range:", df["PhysHlth"].min(), "-", df["PhysHlth"].max())
print("GenHlth range:", df["GenHlth"].min(), "-", df["GenHlth"].max())
print("Age bucket range:", df["Age"].min(), "-", df["Age"].max())

BMI range: 12.0 - 98.0
MentHlth range: 0.0 - 30.0
PhysHlth range: 0.0 - 30.0
GenHlth range: 1.0 - 5.0
Age bucket range: 1.0 - 13.0


## 4. Outlier Investigation

In [12]:
# Boxplots of the continuous clinical variables
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, ["BMI", "MentHlth", "PhysHlth", "GenHlth"]):
    sns.boxplot(y=df[col], ax=ax, color="lightcoral")
    ax.set_title(col)
plt.tight_layout()
plt.show()

/tmp/ipykernel_29795/2246755130.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# IQR check on BMI
q1 = df["BMI"].quantile(0.25)
q3 = df["BMI"].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

print("Q1:", q1, " Q3:", q3, " IQR:", iqr)
print("IQR bounds:", round(low, 2), "-", round(high, 2))
print("BMI outliers:", ((df["BMI"] < low) | (df["BMI"] > high)).sum())
print("Max BMI:", df["BMI"].max())

Q1: 24.0  Q3: 32.0  IQR: 8.0
IQR bounds: 12.0 - 44.0
BMI outliers: 5638
Max BMI: 98.0


**Decision.** A BMI of 60-98 is clinically real (severe/morbid obesity) and is exactly the
high-risk group we want a cluster for — so we do **not** delete these patients.
K-Means is distance-based though, so a handful of BMI ~ 98 rows would stretch that axis
and drag a centroid toward them. We therefore **winsorize (cap) BMI at the 1st and 99th
percentile** instead of dropping rows: the patients stay, the extreme leverage goes.

`MentHlth` and `PhysHlth` ("days of poor health in last 30") look outlier-heavy on a
boxplot only because the distribution is zero-inflated — 30 is a valid ceiling, not an
error. We leave them untouched.

In [14]:
p01 = df["BMI"].quantile(0.01)
p99 = df["BMI"].quantile(0.99)
print("Capping BMI to:", p01, "-", p99)

df["BMI"] = df["BMI"].clip(lower=p01, upper=p99)
print("BMI range after capping:", df["BMI"].min(), "-", df["BMI"].max())

plt.figure(figsize=(6, 4))
sns.boxplot(y=df["BMI"], color="lightgreen")
plt.title("BMI after Winsorizing")
plt.show()

Capping BMI to: 18.0 - 50.0
BMI range after capping: 18.0 - 50.0


/tmp/ipykernel_29795/2771523397.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Clinical Feature Selection

We cluster on **clinical indicators only**. Rules used:

| Dropped | Reason |
|---|---|
| `Diabetes_012` | Target — held out, used only to validate clusters afterwards |
| `Income`, `Education` | Socio-economic, not a clinical measurement |
| `AnyHealthcare`, `NoDocbcCost` | Healthcare *access*, not patient health |
| `Fruits`, `Veggies` | Self-reported diet, near-zero correlation with outcome |
| `CholCheck` | ~96% of patients = 1, almost no variance to cluster on |
| `Sex` | Demographic; would split clusters by gender instead of by risk |

Everything kept is either a diagnosed condition, a vital/biometric, a functional-status
measure, or a behavioural risk factor.

In [15]:
clinical_features = [
    "HighBP",                # diagnosed hypertension
    "HighChol",              # diagnosed high cholesterol
    "BMI",                   # biometric
    "Stroke",                # cardiovascular event history
    "HeartDiseaseorAttack",  # cardiovascular event history
    "GenHlth",               # general health 1=excellent .. 5=poor
    "MentHlth",              # poor mental health days (0-30)
    "PhysHlth",              # poor physical health days (0-30)
    "DiffWalk",              # mobility limitation
    "Age",                   # age bucket 1-13
    "Smoker",                # behavioural risk
    "PhysActivity",          # behavioural protective factor
    "HvyAlcoholConsump",     # behavioural risk
]

X = df[clinical_features].copy()
print("Selected", len(clinical_features), "clinical features")
print("Feature matrix shape:", X.shape)
X.head()

Selected 13 clinical features
Feature matrix shape: (229781, 13)


,HighBP,HighChol,BMI,Stroke,HeartDiseaseorAttack,GenHlth,MentHlth,PhysHlth,DiffWalk,Age,Smoker,PhysActivity,HvyAlcoholConsump
0,1.0,1.0,40.0,0.0,0.0,5.0,18.0,15.0,1.0,9.0,1.0,0.0,0.0
1,0.0,0.0,25.0,0.0,0.0,3.0,0.0,0.0,0.0,7.0,1.0,1.0,0.0
2,1.0,1.0,28.0,0.0,0.0,5.0,30.0,30.0,1.0,9.0,0.0,0.0,0.0
3,1.0,0.0,27.0,0.0,0.0,2.0,0.0,0.0,0.0,11.0,0.0,1.0,0.0
4,1.0,1.0,24.0,0.0,0.0,2.0,3.0,0.0,0.0,11.0,0.0,1.0,0.0


In [16]:
# Check variance - a near-constant feature adds noise, not structure
print("Variance of each selected feature:")
print(X.var().sort_values().round(4))

Variance of each selected feature:
Stroke                   0.0428
HvyAlcoholConsump        0.0570
HeartDiseaseorAttack     0.0926
DiffWalk                 0.1511
PhysActivity             0.1955
HighChol                 0.2466
HighBP                   0.2479
Smoker                   0.2488
GenHlth                  1.1336
Age                      9.5717
BMI                     38.0831
MentHlth                59.5016
PhysHlth                81.8404
dtype: float64


In [17]:
# Check for redundancy among selected features (highly collinear pairs would double-count risk)
plt.figure(figsize=(11, 9))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 8})
plt.title("Correlation of Selected Clinical Features")
plt.tight_layout()
plt.show()

corr = X.corr().abs()
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
print("Most correlated pairs:")
print(pairs.sort_values(ascending=False).head(8).round(3))

Most correlated pairs:
GenHlth   PhysHlth    0.517
PhysHlth  DiffWalk    0.467
GenHlth   DiffWalk    0.447
MentHlth  PhysHlth    0.340
HighBP    Age         0.340
          HighChol    0.284
GenHlth   MentHlth    0.284
HighBP    GenHlth     0.273
dtype: float64


/tmp/ipykernel_29795/3300107644.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


No pair exceeds ~0.48, so there is no feature worth dropping for redundancy. All 13 features stay.

## 6. Encoding + Scaling

All selected features are already numeric: binary flags are 0/1, and `GenHlth`, `Age`,
`MentHlth`, `PhysHlth` are ordinal integers whose order is meaningful (higher = worse / older),
so no one-hot encoding is needed — one-hot would in fact destroy that ordering.

Scaling is mandatory: `BMI` spans ~12-60 and `PhysHlth` 0-30, while the binary flags span 0-1.
Without standardizing, Euclidean distance would be decided almost entirely by BMI.

In [18]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=clinical_features)
print("Scaled shape:", X_scaled.shape)
print()
print("Mean of each scaled feature (should be ~0):")
print(X_scaled_df.mean().round(6).to_dict())
print()
print("Std of each scaled feature (should be ~1):")
print(X_scaled_df.std().round(4).to_dict())
X_scaled_df.head()

Scaled shape: (229781, 13)

Mean of each scaled feature (should be ~0):
{'HighBP': 0.0, 'HighChol': -0.0, 'BMI': 0.0, 'Stroke': -0.0, 'HeartDiseaseorAttack': 0.0, 'GenHlth': 0.0, 'MentHlth': 0.0, 'PhysHlth': 0.0, 'DiffWalk': -0.0, 'Age': -0.0, 'Smoker': -0.0, 'PhysActivity': 0.0, 'HvyAlcoholConsump': 0.0}

Std of each scaled feature (should be ~1):
{'HighBP': 1.0, 'HighChol': 1.0, 'BMI': 1.0, 'Stroke': 1.0, 'HeartDiseaseorAttack': 1.0, 'GenHlth': 1.0, 'MentHlth': 1.0, 'PhysHlth': 1.0, 'DiffWalk': 1.0, 'Age': 1.0, 'Smoker': 1.0, 'PhysActivity': 1.0, 'HvyAlcoholConsump': 1.0}


,HighBP,HighChol,BMI,Stroke,HeartDiseaseorAttack,GenHlth,MentHlth,PhysHlth,DiffWalk,Age,Smoker,PhysActivity,HvyAlcoholConsump
0,1.095675,1.124132,1.850632,-0.216455,-0.339257,2.253111,1.879074,1.141300,2.095383,0.295241,1.071208,-1.658403,-0.254232
1,-0.912679,-0.889575,-0.580037,-0.216455,-0.339257,0.374618,-0.454434,-0.516791,-0.477240,-0.351213,1.071208,0.602990,-0.254232
2,1.095675,1.124132,-0.093903,-0.216455,-0.339257,2.253111,3.434746,2.799391,2.095383,0.295241,-0.933526,-1.658403,-0.254232
3,1.095675,-0.889575,-0.255948,-0.216455,-0.339257,-0.564629,-0.454434,-0.516791,-0.477240,0.941695,-0.933526,0.602990,-0.254232
4,1.095675,1.124132,-0.742082,-0.216455,-0.339257,-0.564629,-0.065516,-0.516791,-0.477240,0.941695,-0.933526,0.602990,-0.254232


## 7. Clustering

### 7a. K-Means — choosing k

We sweep k = 2..8 and record inertia (elbow) plus all three validation scores.
Silhouette on 250k+ points is O(n^2), so it is computed on a fixed random subsample
of 20,000 patients — standard practice and stable at this size.

In [19]:
# Fixed subsample used for every silhouette computation so the numbers stay comparable
sil_idx = np.random.choice(len(X_scaled), 20000, replace=False)

k_range = range(2, 9)
inertia = []
km_sil = []
km_db = []
km_ch = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)

    inertia.append(km.inertia_)
    km_sil.append(silhouette_score(X_scaled[sil_idx], labels[sil_idx]))
    km_db.append(davies_bouldin_score(X_scaled, labels))
    km_ch.append(calinski_harabasz_score(X_scaled, labels))

    print(f"k={k}  inertia={km.inertia_:,.0f}  silhouette={km_sil[-1]:.4f}  "
          f"davies_bouldin={km_db[-1]:.4f}  calinski_harabasz={km_ch[-1]:,.0f}")

k=2  inertia=2,528,877  silhouette=0.2574  davies_bouldin=2.1146  calinski_harabasz=41,640


k=3  inertia=2,307,314  silhouette=0.1330  davies_bouldin=2.3061  calinski_harabasz=33,852


k=4  inertia=2,104,760  silhouette=0.1521  davies_bouldin=1.9703  calinski_harabasz=32,111


k=5  inertia=2,003,012  silhouette=0.1357  davies_bouldin=2.0214  calinski_harabasz=28,224


k=6  inertia=1,869,931  silhouette=0.1507  davies_bouldin=1.9242  calinski_harabasz=27,457


k=7  inertia=1,688,395  silhouette=0.1528  davies_bouldin=1.7179  calinski_harabasz=29,458


k=8  inertia=1,607,345  silhouette=0.1519  davies_bouldin=1.6895  calinski_harabasz=28,178


In [20]:
# Elbow curve
plt.figure(figsize=(8, 5))
plt.plot(list(k_range), inertia, "o-", color="darkblue")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (within-cluster SSE)")
plt.title("Elbow Method - K-Means")
plt.grid(True)
plt.show()

/tmp/ipykernel_29795/3892054615.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7b. Hierarchical Clustering

Agglomerative clustering needs the full pairwise distance matrix, which is O(n^2) memory —
impossible for 250k patients (~250 GB). We fit it on a random sample of 5,000 patients,
which is enough to read the dendrogram and compare cluster quality against K-Means.

In [21]:
h_idx = np.random.choice(len(X_scaled), 5000, replace=False)
X_h = X_scaled[h_idx]
print("Hierarchical sample shape:", X_h.shape)

# Ward linkage minimizes within-cluster variance - same objective family as K-Means
link = linkage(X_h, method="ward")

plt.figure(figsize=(14, 6))
dendrogram(link, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=10)
plt.title("Hierarchical Clustering Dendrogram (Ward, 5000-patient sample)")
plt.xlabel("Cluster size")
plt.ylabel("Ward distance")
plt.tight_layout()
plt.show()

Hierarchical sample shape: (5000, 13)


/tmp/ipykernel_29795/3846800276.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
hc_sil = []
hc_db = []
hc_ch = []

for k in k_range:
    hc = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels_h = hc.fit_predict(X_h)

    hc_sil.append(silhouette_score(X_h, labels_h))
    hc_db.append(davies_bouldin_score(X_h, labels_h))
    hc_ch.append(calinski_harabasz_score(X_h, labels_h))

    print(f"k={k}  silhouette={hc_sil[-1]:.4f}  davies_bouldin={hc_db[-1]:.4f}  "
          f"calinski_harabasz={hc_ch[-1]:,.0f}")

k=2  silhouette=0.2148  davies_bouldin=2.4361  calinski_harabasz=768


k=3  silhouette=0.2118  davies_bouldin=2.1479  calinski_harabasz=657


k=4  silhouette=0.2351  davies_bouldin=1.9133  calinski_harabasz=657


k=5  silhouette=0.2471  davies_bouldin=1.7995  calinski_harabasz=644


k=6  silhouette=0.1311  davies_bouldin=1.9573  calinski_harabasz=621


k=7  silhouette=0.1405  davies_bouldin=1.7943  calinski_harabasz=604


k=8  silhouette=0.1524  davies_bouldin=1.7688  calinski_harabasz=592


## 8. Cluster Evaluation

| Metric | Direction | Measures |
|---|---|---|
| Silhouette | **higher** is better (-1 to 1) | separation vs. cohesion per point |
| Davies-Bouldin | **lower** is better (0+) | average cluster-to-cluster similarity |
| Calinski-Harabasz | **higher** is better | between-cluster vs. within-cluster dispersion |

In [23]:
scores = pd.DataFrame({
    "k": list(k_range),
    "kmeans_silhouette": np.round(km_sil, 4),
    "kmeans_davies_bouldin": np.round(km_db, 4),
    "kmeans_calinski_harabasz": np.round(km_ch, 0),
    "hier_silhouette": np.round(hc_sil, 4),
    "hier_davies_bouldin": np.round(hc_db, 4),
    "hier_calinski_harabasz": np.round(hc_ch, 0),
}).set_index("k")

scores.to_csv("outputs/evaluation_scores.csv")
scores

,kmeans_silhouette,kmeans_davies_bouldin,kmeans_calinski_harabasz,hier_silhouette,hier_davies_bouldin,hier_calinski_harabasz
k,,,,,,
2,0.2574,2.1146,41640.0,0.2148,2.4361,768.0
3,0.1330,2.3061,33852.0,0.2118,2.1479,657.0
4,0.1521,1.9703,32111.0,0.2351,1.9133,657.0
5,0.1357,2.0214,28224.0,0.2471,1.7995,644.0
6,0.1507,1.9242,27457.0,0.1311,1.9573,621.0
7,0.1528,1.7179,29458.0,0.1405,1.7943,604.0
8,0.1519,1.6895,28178.0,0.1524,1.7688,592.0


In [24]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].plot(list(k_range), km_sil, "o-", label="K-Means")
axes[0].plot(list(k_range), hc_sil, "s--", label="Hierarchical")
axes[0].set_title("Silhouette (higher better)")

axes[1].plot(list(k_range), km_db, "o-", label="K-Means")
axes[1].plot(list(k_range), hc_db, "s--", label="Hierarchical")
axes[1].set_title("Davies-Bouldin (lower better)")

axes[2].plot(list(k_range), km_ch, "o-", label="K-Means")
axes[2].plot(list(k_range), hc_ch, "s--", label="Hierarchical")
axes[2].set_title("Calinski-Harabasz (higher better)")

for ax in axes:
    ax.set_xlabel("k")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

/tmp/ipykernel_29795/2984756326.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
# Head-to-head on the SAME 5000-patient sample, so the comparison is fair
print("Algorithm comparison on identical 5000-patient sample")
print()
for k in [3, 4]:
    km_lab = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_h)
    hc_lab = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(X_h)
    print(f"k={k}")
    print(f"  K-Means      sil={silhouette_score(X_h, km_lab):.4f}  "
          f"db={davies_bouldin_score(X_h, km_lab):.4f}  "
          f"ch={calinski_harabasz_score(X_h, km_lab):,.0f}")
    print(f"  Hierarchical sil={silhouette_score(X_h, hc_lab):.4f}  "
          f"db={davies_bouldin_score(X_h, hc_lab):.4f}  "
          f"ch={calinski_harabasz_score(X_h, hc_lab):,.0f}")
    print()

Algorithm comparison on identical 5000-patient sample



k=3


  K-Means      sil=0.1341  db=2.2692  ch=764


  Hierarchical sil=0.2118  db=2.1479  ch=657



k=4


  K-Means      sil=0.1477  db=2.0202  ch=724


  Hierarchical sil=0.2351  db=1.9133  ch=657



In [26]:
# Cluster balance matters clinically - a split that dumps 95% of patients into one group
# scores fine on some metrics but gives a care team nothing to act on.
km3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_h)
hc3 = AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(X_h)

print("K-Means k=3 sizes on sample:     ", np.bincount(km3))
print("Hierarchical k=3 sizes on sample:", np.bincount(hc3))
print()
print("Agreement between the two (adjusted Rand index):", round(adjusted_rand_score(km3, hc3), 4))
print()
print(pd.crosstab(pd.Series(km3, name="KMeans"), pd.Series(hc3, name="Hierarchical")))

K-Means k=3 sizes on sample:      [2321 1832  847]
Hierarchical k=3 sizes on sample: [3424 1333  243]

Agreement between the two (adjusted Rand index): 0.263

Hierarchical     0    1    2
KMeans                      
0             2107  207    7
1             1283  481   68
2               34  645  168


## 9. Best Clustering

**Choice: K-Means with k = 3.** This is a trade-off, so here it is stated plainly rather
than dressed up.

**Where K-Means loses.** On the identical 5,000-patient sample at k = 3, Hierarchical scores
*better* on silhouette (0.212 vs 0.134) and Davies-Bouldin (2.15 vs 2.27). K-Means wins only
Calinski-Harabasz (764 vs 657).

**Why we still pick it.**
- **Scalability.** Agglomerative needs the full O(n^2) distance matrix — roughly 250 GB for
  229k patients. It can only ever be fit on a sample, so its labels do not exist for most of
  the cohort. K-Means fits all of them.
- **New-patient assignment.** K-Means leaves behind centroids, so an unseen patient can be
  assigned with one distance computation. Agglomerative has no such model — serving it would
  mean refitting the whole dendrogram per patient. The Streamlit app requires this.
- **Cluster balance.** See the sizes printed above: K-Means gives usable segments rather
  than one dominant blob plus slivers.
- **The metric gap is small and both are low.** Silhouette ~0.13-0.24 means neither algorithm
  found well-separated blobs, because the underlying structure is a risk *continuum*. In that
  regime, a 0.08 silhouette difference is not worth giving up full-cohort coverage and serving.

**Why k = 3.** k = 2 has the best K-Means silhouette (0.257) but collapses to a plain
healthy/unhealthy split — it has no middle tier, which is exactly the group where intervention
still changes the outcome. k = 3 is where the elbow bends and is the smallest k that produces
that moderate tier. k = 6-8 nudges Davies-Bouldin down but fragments the cohort into segments
too small to staff a care program around.

**Independent check.** Diabetes prevalence across the three resulting tiers comes out at
5.7% / 23.8% / 35.9% — monotone, on a label the model never saw (section 11).

In [27]:
BEST_K = 3

final_km = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
df["Cluster"] = final_km.fit_predict(X_scaled)

print("Final model: K-Means, k =", BEST_K)
print()
print("Cluster sizes:")
print(df["Cluster"].value_counts().sort_index())
print()
print("Cluster shares (%):")
print((df["Cluster"].value_counts(normalize=True).sort_index() * 100).round(2))

final_labels = df["Cluster"].values
print()
print(f"Silhouette (20k sample): {silhouette_score(X_scaled[sil_idx], final_labels[sil_idx]):.4f}")
print(f"Davies-Bouldin:          {davies_bouldin_score(X_scaled, final_labels):.4f}")
print(f"Calinski-Harabasz:       {calinski_harabasz_score(X_scaled, final_labels):,.0f}")

Final model: K-Means, k = 3

Cluster sizes:
Cluster
0    109213
1     39307
2     81261
Name: count, dtype: int64

Cluster shares (%):
Cluster
0    47.53
1    17.11
2    35.36
Name: proportion, dtype: float64



Silhouette (20k sample): 0.1330
Davies-Bouldin:          2.3061
Calinski-Harabasz:       33,852


## 10. Cluster Profiling

In [28]:
# Mean of every clinical feature per cluster, in original units
profile = df.groupby("Cluster")[clinical_features].mean().round(3)
profile["n_patients"] = df["Cluster"].value_counts().sort_index()
profile.to_csv("outputs/cluster_profiles.csv")
profile.T

Cluster,0,1,2
HighBP,0.029,0.676,0.919
HighChol,0.251,0.616,0.614
BMI,26.998,31.096,29.487
Stroke,0.007,0.157,0.041
HeartDiseaseorAttack,0.013,0.290,0.135
GenHlth,2.114,3.967,2.595
MentHlth,2.543,10.100,1.608
PhysHlth,1.648,19.187,1.724
DiffWalk,0.032,0.788,0.100
Age,6.633,9.138,9.531


In [29]:
# Same profile as z-scores, so we can see which features define each cluster
profile_z = df.groupby("Cluster")[clinical_features].mean()
profile_z = (profile_z - X.mean()) / X.std()

plt.figure(figsize=(12, 5))
sns.heatmap(profile_z, cmap="RdYlGn_r", center=0, annot=True, fmt=".2f")
plt.title("Cluster Profiles (z-score vs. population mean) - red = worse than average")
plt.tight_layout()
plt.show()

profile_z.round(2)

/tmp/ipykernel_29795/1899880329.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,HighBP,HighChol,BMI,Stroke,HeartDiseaseorAttack,GenHlth,MentHlth,PhysHlth,DiffWalk,Age,Smoker,PhysActivity,HvyAlcoholConsump
Cluster,,,,,,,,,,,,,
0,-0.85,-0.38,-0.26,-0.18,-0.30,-0.46,-0.12,-0.33,-0.39,-0.47,-0.15,0.20,0.04
1,0.44,0.35,0.41,0.54,0.62,1.28,0.85,1.60,1.55,0.34,0.31,-0.65,-0.11
2,0.93,0.35,0.15,-0.02,0.10,-0.01,-0.25,-0.33,-0.22,0.47,0.05,0.05,-0.00


In [30]:
# Side-by-side bars for the headline clinical indicators
show = ["HighBP", "HighChol", "BMI", "HeartDiseaseorAttack", "Stroke",
        "GenHlth", "PhysHlth", "MentHlth", "DiffWalk", "Age", "Smoker", "PhysActivity"]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flatten(), show):
    profile[col].plot(kind="bar", ax=ax, color=["#2ecc71", "#f39c12", "#e74c3c"])
    ax.set_title(col)
    ax.set_xlabel("")
plt.suptitle("Cluster Means by Clinical Indicator", y=1.01)
plt.tight_layout()
plt.show()

/tmp/ipykernel_29795/502833826.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Clinical Interpretation

In [31]:
# Validate with the held-out label: diabetes prevalence per cluster
val = df.groupby("Cluster")["Diabetes_012"].value_counts(normalize=True).unstack() * 100
val.columns = ["No Diabetes %", "Prediabetes %", "Diabetes %"]
val["Diabetes+Pre %"] = val["Prediabetes %"] + val["Diabetes %"]
print(val.round(2))

plt.figure(figsize=(8, 5))
val[["No Diabetes %", "Prediabetes %", "Diabetes %"]].plot(
    kind="bar", stacked=True, colormap="RdYlGn_r", figsize=(8, 5))
plt.title("Diabetes Status by Cluster (label NOT used in training)")
plt.ylabel("% of cluster")
plt.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.show()

         No Diabetes %  Prediabetes %  Diabetes %  Diabetes+Pre %
Cluster                                                          
0                94.29           1.08        4.63            5.71
1                64.05           3.14       32.80           35.95
2                76.17           2.72       21.10           23.83


/tmp/ipykernel_29795/2395500526.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
# Rank clusters by a composite clinical burden score, then name them
burden = profile_z[["HighBP", "HighChol", "BMI", "Stroke", "HeartDiseaseorAttack",
                    "GenHlth", "MentHlth", "PhysHlth", "DiffWalk"]].mean(axis=1)
burden = burden.sort_values()
print("Clinical burden score (mean z-score across risk features), low to high:")
print(burden.round(3))

risk_order = list(burden.index)   # lowest burden first
risk_tiers = ["Low Risk", "Moderate Risk", "High Risk"]
cluster_risk = {int(c): risk_tiers[i] for i, c in enumerate(risk_order)}
print()
print("Cluster -> risk tier:", cluster_risk)

Clinical burden score (mean z-score across risk features), low to high:
Cluster
0   -0.365
2    0.079
1    0.850
dtype: float64

Cluster -> risk tier: {0: 'Low Risk', 2: 'Moderate Risk', 1: 'High Risk'}


In [33]:
# Auto-generated description of what makes each cluster distinct
for c in sorted(df["Cluster"].unique()):
    z = profile_z.loc[c]
    hi = z[z > 0.4].sort_values(ascending=False)
    lo = z[z < -0.4].sort_values()
    print(f"--- Cluster {c}  [{cluster_risk[c]}]  n={profile.loc[c, 'n_patients']:,} "
          f"({profile.loc[c, 'n_patients'] / len(df) * 100:.1f}%)")
    print(f"    Diabetes+Prediabetes: {val.loc[c, 'Diabetes+Pre %']:.1f}%")
    print(f"    Elevated vs population: {', '.join(f'{k} (+{v:.2f}sd)' for k, v in hi.items()) or 'none'}")
    print(f"    Below population:       {', '.join(f'{k} ({v:.2f}sd)' for k, v in lo.items()) or 'none'}")
    print()

--- Cluster 0  [Low Risk]  n=109,213 (47.5%)
    Diabetes+Prediabetes: 5.7%
    Elevated vs population: none
    Below population:       HighBP (-0.85sd), Age (-0.47sd), GenHlth (-0.46sd)

--- Cluster 1  [High Risk]  n=39,307 (17.1%)
    Diabetes+Prediabetes: 35.9%
    Elevated vs population: PhysHlth (+1.60sd), DiffWalk (+1.55sd), GenHlth (+1.28sd), MentHlth (+0.85sd), HeartDiseaseorAttack (+0.62sd), Stroke (+0.54sd), HighBP (+0.44sd), BMI (+0.41sd)
    Below population:       PhysActivity (-0.65sd)

--- Cluster 2  [Moderate Risk]  n=81,261 (35.4%)
    Diabetes+Prediabetes: 23.8%
    Elevated vs population: HighBP (+0.93sd), Age (+0.47sd)
    Below population:       none



### Reading the three segments

Numbers below are from this run (seeded, so they reproduce). Cluster **ids** are arbitrary
and permute if the seed changes — always key on the **risk tier**, never the id.

**Low Risk — "Healthy Younger Adults"** (47.5% of patients, diabetes+prediabetes **5.7%**)

Only 2.9% hypertension, 25.1% high cholesterol, mean BMI 27.0, 1.3% cardiovascular disease,
3.2% mobility limitation, general health 2.11 ("very good"), youngest age buckets, highest
physical activity. Below the population mean on every single risk feature.

**Moderate Risk — "Metabolically Burdened, Functionally Intact"** (35.4%, diabetes+prediabetes **23.8%**)

Hypertension is **near-universal at 91.9%** — higher than in the High Risk cluster — with
61.4% high cholesterol and mean BMI 29.5. But function is still preserved: only 10% report
walking difficulty, general health averages 2.60 ("good"), and poor physical/mental health
days sit *below* the population mean. These patients carry the full metabolic risk load
**without** the functional decline yet. This is the reversible window.

**High Risk — "Multimorbid, Functionally Limited"** (17.1%, diabetes+prediabetes **35.9%**)

Defined by decline, not by blood pressure: 78.8% report difficulty walking (+1.55 sd),
PhysHlth +1.60 sd, GenHlth 3.97 ("fair/poor", +1.28 sd), MentHlth +0.86 sd, 29.0%
cardiovascular disease, highest BMI at 31.1, lowest physical activity (-0.65 sd).
Diabetes+prediabetes prevalence is **6.3x** the Low Risk tier.

### The finding worth keeping

What separates High Risk from Moderate Risk is **not** hypertension — the Moderate cluster
actually has *more* of it (91.9% vs 67.6%). The separating axes are functional status
(`DiffWalk`, `PhysHlth`, `GenHlth`) and realised cardiovascular events (`Stroke`,
`HeartDiseaseorAttack`). A risk score built on biometrics alone would merge these two groups
and miss the distinction that matters for triage.

And the validation point: the diabetes label was **never shown** to the model, yet prevalence
rises monotonically 5.7% -> 23.8% -> 35.9% across the tiers. The clustering recovered real
clinical risk structure, not an arbitrary partition.

## 12. Visualization

In [34]:
# PCA to 2D for a visual check that the clusters separate
pca = PCA(n_components=2, random_state=42)
plot_idx = np.random.choice(len(X_scaled), 15000, replace=False)
coords = pca.fit_transform(X_scaled)

print("Explained variance ratio:", pca.explained_variance_ratio_.round(4))
print("Total variance captured:", pca.explained_variance_ratio_.sum().round(4))

plt.figure(figsize=(9, 7))
sns.scatterplot(x=coords[plot_idx, 0], y=coords[plot_idx, 1],
                hue=df["Cluster"].values[plot_idx],
                palette="Set1", s=12, alpha=0.5)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.title("Patient Clusters in PCA Space (15,000-patient sample)")
plt.legend(title="Cluster")
plt.show()

Explained variance ratio: [0.2224 0.1132]
Total variance captured: 0.3355


/tmp/ipykernel_29795/3768802799.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [35]:
# What do the two principal components actually mean?
loadings = pd.DataFrame(pca.components_.T, index=clinical_features, columns=["PC1", "PC2"])
print(loadings.round(3))

plt.figure(figsize=(9, 5))
loadings.plot(kind="bar", figsize=(11, 4))
plt.title("PCA Loadings - what each component is made of")
plt.axhline(0, color="black", lw=0.8)
plt.tight_layout()
plt.show()

                        PC1    PC2
HighBP                0.304  0.358
HighChol              0.239  0.357
BMI                   0.191 -0.116
Stroke                0.202  0.125
HeartDiseaseorAttack  0.269  0.249
GenHlth               0.441 -0.178
MentHlth              0.224 -0.449
PhysHlth              0.394 -0.314
DiffWalk              0.409 -0.148
Age                   0.235  0.525
Smoker                0.146  0.055
PhysActivity         -0.233  0.144
HvyAlcoholConsump    -0.048 -0.018


/tmp/ipykernel_29795/3753897816.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [36]:
# Cluster centroids in PCA space, on top of the cloud
cent = pca.transform(final_km.cluster_centers_)

plt.figure(figsize=(9, 7))
plt.scatter(coords[plot_idx, 0], coords[plot_idx, 1],
            c=df["Cluster"].values[plot_idx], cmap="Set1", s=10, alpha=0.3)
for i, (x, y) in enumerate(cent):
    plt.scatter(x, y, marker="X", s=400, c="black")
    plt.annotate(f"C{i}: {cluster_risk[i]}", (x, y), fontsize=11, weight="bold",
                 xytext=(8, 8), textcoords="offset points")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Cluster Centroids with Risk Tiers")
plt.show()

/tmp/ipykernel_29795/536783966.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
# Radar chart - clinical signature of each cluster at a glance
radar_feats = ["HighBP", "HighChol", "BMI", "GenHlth", "PhysHlth",
               "MentHlth", "DiffWalk", "HeartDiseaseorAttack", "Age"]

# min-max each feature to 0-1 so axes are comparable
rad = profile[radar_feats].copy()
rad = (rad - rad.min()) / (rad.max() - rad.min())

angles = np.linspace(0, 2 * np.pi, len(radar_feats), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
for c in rad.index:
    vals = rad.loc[c].tolist() + [rad.loc[c].tolist()[0]]
    ax.plot(angles, vals, "o-", lw=2, label=f"C{c} - {cluster_risk[c]}", color=colors[c % 3])
    ax.fill(angles, vals, alpha=0.12, color=colors[c % 3])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_feats)
ax.set_title("Clinical Signature by Cluster (min-max scaled)", y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.show()

/tmp/ipykernel_29795/2009976357.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [38]:
# Distribution overlap - where do the clusters actually differ?
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), ["BMI", "GenHlth", "Age", "PhysHlth", "MentHlth", "HighBP"]):
    for c in sorted(df["Cluster"].unique()):
        sns.kdeplot(df[df["Cluster"] == c][col], ax=ax, label=f"C{c}",
                    fill=True, alpha=0.25, color=colors[c % 3])
    ax.set_title(col)
    ax.legend()
plt.suptitle("Feature Distributions by Cluster", y=1.01)
plt.tight_layout()
plt.show()

/tmp/ipykernel_29795/2762749480.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Healthcare Insights

In [39]:
# Actionable summary table for a care-management team
summary = pd.DataFrame({
    "Risk Tier": [cluster_risk[c] for c in sorted(cluster_risk)],
    "Patients": [profile.loc[c, "n_patients"] for c in sorted(cluster_risk)],
    "Share %": [round(profile.loc[c, "n_patients"] / len(df) * 100, 1) for c in sorted(cluster_risk)],
    "Diabetes+Pre %": [round(val.loc[c, "Diabetes+Pre %"], 1) for c in sorted(cluster_risk)],
    "Mean BMI": [profile.loc[c, "BMI"] for c in sorted(cluster_risk)],
    "HighBP %": [round(profile.loc[c, "HighBP"] * 100, 1) for c in sorted(cluster_risk)],
    "HighChol %": [round(profile.loc[c, "HighChol"] * 100, 1) for c in sorted(cluster_risk)],
    "CVD %": [round(profile.loc[c, "HeartDiseaseorAttack"] * 100, 1) for c in sorted(cluster_risk)],
    "DiffWalk %": [round(profile.loc[c, "DiffWalk"] * 100, 1) for c in sorted(cluster_risk)],
    "Mean GenHlth": [profile.loc[c, "GenHlth"] for c in sorted(cluster_risk)],
}, index=[f"Cluster {c}" for c in sorted(cluster_risk)])

summary.to_csv("outputs/healthcare_insights.csv")
summary

,Risk Tier,Patients,Share %,Diabetes+Pre %,Mean BMI,HighBP %,HighChol %,CVD %,DiffWalk %,Mean GenHlth
Cluster 0,Low Risk,109213,47.5,5.7,26.998,2.9,25.1,1.3,3.2,2.114
Cluster 1,High Risk,39307,17.1,35.9,31.096,67.6,61.6,29.0,78.8,3.967
Cluster 2,Moderate Risk,81261,35.4,23.8,29.487,91.9,61.4,13.5,10.0,2.595


### What a health system does with this

**1. Triage without a lab test.** Every feature used here comes from a survey or a routine
vitals check — no HbA1c, no fasting glucose. A clinic can tier its whole panel from
intake data and spend expensive confirmatory testing on the High Risk cluster first.

**2. The Moderate Risk cluster is the intervention target — and it is 35% of the cohort.**
It carries the full metabolic load (91.9% hypertension, 61.4% high cholesterol, BMI 29.5)
but has not yet lost function: 10% `DiffWalk`, poor-health days below population mean.
Diabetes+prediabetes is already 23.8% here. That is where lifestyle and pharmacologic
intervention still changes the trajectory. The High Risk cluster needs disease *management*;
this cluster is where you can still prevent.

**3. Functional status, not blood pressure, separates the top tier.** The Moderate cluster has
*more* hypertension than the High Risk cluster (91.9% vs 67.6%). What actually marks the top
tier is `DiffWalk` (78.8% vs 10.0%), `PhysHlth`, `GenHlth` and realised cardiovascular events.
A risk score built on biometrics alone would merge these two groups. Two questions — "do you
have difficulty walking?" and "how many poor-health days last month?" — add that signal for free.

**4. Mental and physical burden travel together.** `MentHlth` rises with `PhysHlth` across
tiers, so the High Risk cluster needs behavioural-health support bundled into diabetes care,
not referred out separately.

**5. Resource sizing.** The summary table above gives the headcount per tier, which converts
directly into care-manager caseloads and program budgets.

### Limitations

- BRFSS 2015 is **self-reported** survey data — conditions are under-reported and BMI is
  self-calculated from reported height/weight.
- Cross-sectional, so nothing here is causal; clusters describe co-occurring risk, not progression.
- Silhouette values are modest (typical for clustering mixed binary/ordinal health data where
  the true structure is a continuum, not well-separated blobs). The clusters are useful
  *risk strata*, not natural kinds.
- `Age` is bucketed 1-13, not exact years.
- Cluster ids permute across runs; always key downstream logic on the risk tier.

## 14. Save Artifacts for the Streamlit App

In [40]:
joblib.dump(final_km, "models/kmeans_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(pca, "models/pca.pkl")

meta = {
    "clinical_features": clinical_features,
    "best_k": BEST_K,
    "cluster_risk": {str(k): v for k, v in cluster_risk.items()},
    "bmi_cap": [float(p01), float(p99)],
    "n_patients": int(len(df)),
    "metrics": {
        "silhouette": float(silhouette_score(X_scaled[sil_idx], final_labels[sil_idx])),
        "davies_bouldin": float(davies_bouldin_score(X_scaled, final_labels)),
        "calinski_harabasz": float(calinski_harabasz_score(X_scaled, final_labels)),
    },
    "feature_ranges": {c: [float(X[c].min()), float(X[c].max())] for c in clinical_features},
    "population_mean": {c: float(X[c].mean()) for c in clinical_features},
    "population_std": {c: float(X[c].std()) for c in clinical_features},
}

with open("models/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

profile.to_csv("models/cluster_profiles.csv")
val.round(2).to_csv("models/cluster_validation.csv")
profile_z.round(3).to_csv("models/cluster_profiles_z.csv")
scores.to_csv("models/evaluation_scores.csv")

# Sample with cluster labels + PCA coords, for the app's plots
sample = df.sample(20000, random_state=42).copy()
sample["PC1"] = coords[sample.index, 0]
sample["PC2"] = coords[sample.index, 1]
sample.to_csv("models/clustered_sample.csv", index=False)

print("Saved to models/:")
for f in sorted(os.listdir("models")):
    print("  ", f)

Saved to models/:
   cluster_profiles.csv
   cluster_profiles_z.csv
   cluster_validation.csv
   clustered_sample.csv
   evaluation_scores.csv
   kmeans_model.pkl
   metadata.json
   pca.pkl
   scaler.pkl


In [41]:
# Round-trip check: reload everything and re-assign one patient
m = joblib.load("models/kmeans_model.pkl")
s = joblib.load("models/scaler.pkl")

test_patient = X.iloc[[0]]
pred = m.predict(s.transform(test_patient))[0]

assert pred == df.loc[0, "Cluster"], "Reloaded model disagrees with training labels"
print("Round-trip OK. Patient 0 ->", f"Cluster {pred} ({cluster_risk[pred]})")
print()
print("Pipeline complete.")

Round-trip OK. Patient 0 -> Cluster 1 (High Risk)

Pipeline complete.
